# Teil 4 - Evaluation

In diesem Notebook bewerte ich das trainierte Klassifikationsmodell. Ich bestimme wichtige Felder, berechne passende Metriken, erstelle eine Wahrheitsmatrix und leite daraus Sensitivität sowie Spezifität ab.

## 1. Modell erneut vorbereiten

Damit dieses Notebook unabhängig lauffähig ist, lade ich den Datensatz erneut, bilde dieselbe Zielvariable und trainiere dasselbe Modell mit derselben Train-/Test-Aufteilung wie in `model.ipynb`.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DATA_FILE = Path('Smartphone_Usage_Productivity_Dataset_50000.csv')
df = pd.read_csv(DATA_FILE, sep=';')
df['High_Stress'] = (df['Stress_Level'] >= 7).astype(int)

X = df.drop(columns=['User_ID', 'Stress_Level', 'High_Stress'])
y = df['High_Stress']

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [column for column in X.columns if column not in numeric_features]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

model = RandomForestClassifier(
    n_estimators=120,
    max_depth=6,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

clf = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', model)
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]
print('Evaluation vorbereitet.')

Evaluation vorbereitet.


## 2. Aussagekräftige Felder bestimmen

Der Random Forest berechnet Feature Importances. Kategoriale Spalten werden durch One-Hot-Encoding in mehrere Modellspalten zerlegt. Deshalb fasse ich diese Importances wieder auf die ursprünglichen Felder zusammen.

In [2]:
encoded_feature_names = clf.named_steps['preprocess'].get_feature_names_out()
importances = clf.named_steps['model'].feature_importances_

importance_df = pd.DataFrame({
    'encoded_feature': encoded_feature_names,
    'importance': importances
})

def original_feature_name(encoded_name):
    if encoded_name.startswith('num__'):
        return encoded_name.replace('num__', '', 1)
    if encoded_name.startswith('cat__'):
        remainder = encoded_name.replace('cat__', '', 1)
        for column in categorical_features:
            if remainder.startswith(column + '_'):
                return column
        return remainder
    return encoded_name

importance_df['feature'] = importance_df['encoded_feature'].apply(original_feature_name)
feature_importance = (
    importance_df.groupby('feature', as_index=False)['importance']
    .sum()
    .sort_values('importance', ascending=False)
)

feature_importance.round(4)

,feature,importance
3,Daily_Phone_Hours,0.1460
8,Social_Media_Hours,0.1388
9,Weekend_Screen_Time_Hours,0.1340
7,Sleep_Hours,0.1293
1,App_Usage_Count,0.1207
0,Age,0.1098
10,Work_Productivity_Score,0.0593
2,Caffeine_Intake_Cups,0.0523
6,Occupation,0.0484
5,Gender,0.0382


In [3]:
plt.figure(figsize=(8, 4))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.gca().invert_yaxis()
plt.title('Wichtige Felder laut Random Forest')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.show()

/var/folders/hp/m0vq6vz143s1lhd11rqb_ft00000gn/T/ipykernel_25798/1935581696.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Passende Messmetriken

Da `High_Stress` eine binäre Klassifikation ist, passen Accuracy, Precision, Recall/Sensitivität und F1-Score. Accuracy allein reicht hier nicht, weil die Klasse `kein hoher Stress` häufiger ist. Darum vergleiche ich auch mit einer einfachen Basislinie, die immer `kein hoher Stress` vorhersagt.

In [4]:
baseline_pred = np.zeros_like(y_test)

metrics = pd.DataFrame([
    {
        'Modell': 'RandomForestClassifier',
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall/Sensitivität': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
    },
    {
        'Modell': 'Basislinie: immer kein hoher Stress',
        'Accuracy': accuracy_score(y_test, baseline_pred),
        'Precision': precision_score(y_test, baseline_pred, zero_division=0),
        'Recall/Sensitivität': recall_score(y_test, baseline_pred),
        'F1-Score': f1_score(y_test, baseline_pred),
    }
])

metrics.round(3)

,Modell,Accuracy,Precision,Recall/Sensitivität,F1-Score
0,RandomForestClassifier,0.514,0.409,0.489,0.446
1,Basislinie: immer kein hoher Stress,0.601,0.000,0.000,0.000


## 4. Wahrheitsmatrix, Sensitivität und Spezifität

Die Wahrheitsmatrix vergleicht echte Klassen mit vorhergesagten Klassen. Sensitivität misst, wie viele echte High-Stress-Fälle erkannt werden. Spezifität misst, wie viele Nicht-High-Stress-Fälle korrekt als negativ erkannt werden.

In [5]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

confusion_table = pd.DataFrame(
    cm,
    index=['echt: kein hoher Stress', 'echt: hoher Stress'],
    columns=['vorhergesagt: kein hoher Stress', 'vorhergesagt: hoher Stress']
)

print('Sensitivität:', round(sensitivity, 3))
print('Spezifität:', round(specificity, 3))
confusion_table

Sensitivität: 0.489
Spezifität: 0.531


,vorhergesagt: kein hoher Stress,vorhergesagt: hoher Stress
echt: kein hoher Stress,3193,2817
echt: hoher Stress,2039,1951


In [6]:
plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap='Blues')
plt.title('Wahrheitsmatrix')
plt.xticks([0, 1], ['kein hoher Stress', 'hoher Stress'], rotation=30, ha='right')
plt.yticks([0, 1], ['kein hoher Stress', 'hoher Stress'])
plt.xlabel('Vorhersage')
plt.ylabel('Echter Wert')

for row in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        plt.text(col, row, cm[row, col], ha='center', va='center')

plt.tight_layout()
plt.show()

/var/folders/hp/m0vq6vz143s1lhd11rqb_ft00000gn/T/ipykernel_25798/907434178.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Interpretation der Evaluation

Das Modell funktioniert technisch korrekt, erkennt aber nur schwache Muster. Die Accuracy liegt unter der einfachen Basislinie, während Sensitivität und Spezifität ungefähr ausgeglichener sind. Das ist plausibel, weil die explorative Analyse fast keine Korrelationen zwischen Stress und den verfügbaren Merkmalen gezeigt hat. Für reale Entscheidungen wäre das Modell ungeeignet; für die LB zeigt es aber den kompletten ML-Ablauf und eine ehrliche Bewertung der Datenqualität.